<a href="https://colab.research.google.com/github/Shashank14081987/Deep-Learning/blob/main/Mask_unmask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from sklearn.metrics import classification_report

In [ ]:
# STEP 2: Unzip and Explore Dataset
from google.colab import files
uploaded = files.upload()  # Upload archive (1).zip manually

with zipfile.ZipFile('archive (1).zip', 'r') as zip_ref:
    zip_ref.extractall('face_mask_dataset')

base_dir = 'face_mask_dataset/Face Mask Dataset'
print("Classes:", os.listdir(base_dir + '/Train'))

In [ ]:
# STEP 3: Visualize Sample Images
import matplotlib.image as mpimg

def plot_images(class_name, path='Train', num=5):
    class_path = os.path.join(base_dir, path, class_name)
    plt.figure(figsize=(15, 5))
    for i, img_name in enumerate(os.listdir(class_path)[:num]):
        img = mpimg.imread(os.path.join(class_path, img_name))
        plt.subplot(1, num, i+1)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis('off')
    plt.show()

plot_images('WithMask')
plot_images('WithoutMask')

In [ ]:
# STEP 4: Data Generators
train_dir = os.path.join(base_dir, 'Train')
val_dir = os.path.join(base_dir, 'Validation')
test_dir = os.path.join(base_dir, 'Test')

train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir, target_size=(128, 128), class_mode='binary', batch_size=32)
val_generator = val_test_datagen.flow_from_directory(val_dir, target_size=(128, 128), class_mode='binary', batch_size=32)
test_generator = val_test_datagen.flow_from_directory(test_dir, target_size=(128, 128), class_mode='binary', batch_size=32, shuffle=False)

In [ ]:
# STEP 5: Model From Scratch
model_scratch = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_scratch.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_scratch = model_scratch.fit(train_generator, epochs=10, validation_data=val_generator)

In [ ]:
 STEP 6: Pretrained Model (MobileNetV2)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128,128,3))
base_model.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model_pretrained = Model(inputs=base_model.input, outputs=predictions)
model_pretrained.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_pretrained = model_pretrained.fit(train_generator, epochs=10, validation_data=val_generator)

In [ ]:
# STEP 7: Data Augmentation + Pretrained Model
augmented_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

augmented_train_generator = augmented_datagen.flow_from_directory(train_dir, target_size=(128, 128), class_mode='binary', batch_size=32)

model_aug = Model(inputs=base_model.input, outputs=predictions)
model_aug.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_aug = model_aug.fit(augmented_train_generator, epochs=10, validation_data=val_generator)

In [ ]:
# STEP 8: Evaluate All Models
print("Model from Scratch:")
model_scratch.evaluate(test_generator)

print("\nPretrained Model:")
model_pretrained.evaluate(test_generator)

print("\nPretrained with Augmentation:")
model_aug.evaluate(test_generator)